# RAG für GitHub Issues mit Gemini API & FAISS (Drive Storage)

Dieses Notebook ist eine angepasste Version des Hugging Face Cookbooks. Es nutzt die **Google Gemini API** für die Generierung und speichert den **FAISS-Index dauerhaft in Google Drive**, um Zeit und Ressourcen zu sparen.

In [ ]:
# Installation der benötigten Bibliotheken
!pip install -q -U langchain-community langchain-huggingface langchain-google-genai faiss-cpu sentence-transformers

In [ ]:
from google.colab import drive
import os

# Google Drive mounten
drive.mount('/content/drive')

# Pfad für die Vektordatenbank definieren
DB_FAISS_PATH = '/content/drive/MyDrive/colab_data/faiss_index_peft'

## Daten vorbereiten
Wir laden die Issues aus dem PEFT-Repository. Du benötigst einen GitHub Token.

In [ ]:
from getpass import getpass
ACCESS_TOKEN = getpass("YOUR_GITHUB_PERSONAL_TOKEN")

Für später brauchen wir auch den Google AI API Key

In [ ]:
GOOGLE_API_KEY = getpass("YOUR_GOOGLE_AI_API_KEY")

In [ ]:
from langchain_community.document_loaders import GitHubIssuesLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Falls die DB nicht existiert, müssen wir laden und splitten
if not os.path.exists(DB_FAISS_PATH):
    loader = GitHubIssuesLoader(
        repo="huggingface/peft",
        access_token=ACCESS_TOKEN,
        include_prs=False,
        state="all"
    )
    docs = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=30)
    chunked_docs = splitter.split_documents(docs)
    print(f"{len(chunked_docs)} Chunks erstellt.")

## FAISS Vektordatenbank (Laden oder Erstellen)
Hier prüfen wir, ob wir die DB aus dem Drive laden können oder neu erstellen müssen.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name='BAAI/bge-base-en-v1.5')

if os.path.exists(DB_FAISS_PATH):
    print("Lade existierende FAISS Datenbank aus Google Drive...")
    db = FAISS.load_local(DB_FAISS_PATH, embeddings, allow_dangerous_deserialization=True)
else:
    print("Erstelle neue FAISS Datenbank und speichere in Drive...")
    db = FAISS.from_documents(chunked_docs, embeddings)
    os.makedirs(os.path.dirname(DB_FAISS_PATH), exist_ok=True)
    db.save_local(DB_FAISS_PATH)

retriever = db.as_retriever(search_type="similarity", search_kwargs={'k': 4})

## Setup Gemini API
Stelle sicher, dass dein `GOOGLE_API_KEY` in den Colab Secrets gespeichert ist.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.2)

# Prompt Templates
simple_prompt = ChatPromptTemplate.from_messages([("human", "{question}")])
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Beantworte die Frage basierend auf dem Kontext:\n\n{context}"),
    ("human", "{question}")
])

llm_chain = simple_prompt | llm | StrOutputParser()
rag_chain = ({"context": retriever, "question": RunnablePassthrough()} | rag_prompt | llm | StrOutputParser())

## Vergleich der Ergebnisse
Wir vergleichen die Antwort ohne Kontext mit der Antwort, die durch RAG informiert wurde.

In [ ]:
question = "How do you combine multiple adapters?"

print("--- OHNE RAG ---")
print(llm_chain.invoke({"question": question}))

print("\n" + "="*50 + "\n")

print("--- MIT RAG ---")
print(rag_chain.invoke(question))